# Semantic Kernel 

In this code sample, you will use the [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI Framework to create a basic agent. 

The goal of this sample is to show you the steps that we will later use in the additional code samples when implementing the different agentic patterns. 

## Import the Needed Python Packages 

In [ ]:
import os 
from typing import Annotated
from openai import AsyncAzureOpenAI

from dotenv import load_dotenv

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function
from azure.core.credentials import AzureKeyCredential

## Creating the Client

In this sample, we will use [Azure OpenAI](https://azure.microsoft.com/products/ai-services/openai-service) for access to the LLM. 

The deployment is configured to use `gpt-4o`. You can change the deployment name to another model deployment available in your Azure OpenAI resource to see different results. 

We use the `AzureOpenAIChatCompletion` connector within Semantic Kernel specifically designed for Azure OpenAI services. There are also other [available connectors](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) to use Semantic Kernel for other model providers.

In [27]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [33]:
load_dotenv()
# Load environment variables from .env file
print("Loading environment variables from .env file...")

# Print relevant environment variables (without exposing sensitive data)
azure_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
model_name = os.getenv("AZURE_AI_FOUNDRY_MODEL")
api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")

print(f"Azure Endpoint: {azure_endpoint}")
print(f"API Version: {api_version}")
print(f"Model Name: {model_name}")


# Azure OpenAI Configuration
client = AsyncAzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version,  # or latest API version
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id=model_name,  # Your deployment name
    async_client=client
)

Loading environment variables from .env file...
Azure Endpoint: https://ibeljan-foundry.openai.azure.com/
API Version: 2024-02-01
Model Name: gpt-4o


## Creating the Agent 

Below we create the Agent called `TravelAgent`.

For this example, we are using very simple instructions. You can change these instructions to see how the agent responds differently. 

In [34]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## Running the Agent

Now we can run the Agent by defining a thread of type `ChatHistoryAgentThread`.  Any required system messages are provided to the agent's invoke_stream `messages` keyword argument.

After these are defined, we create a `user_inputs` that will be what the user is sending to the agent. In this case, we have set this message to `Plan me a sunny vacation`. 

Feel free to change this message to see how the agent responds differently. 

In [35]:
async def main():
    # Create a new thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    thread: ChatHistoryAgentThread | None = None

    user_inputs = [
        "Plan me a day trip.",
    ]

    for user_input in user_inputs:
        print(f"# User: {user_input}\n")
        first_chunk = True
        async for response in agent.invoke_stream(
            messages=user_input, thread=thread,
        ):
            # 5. Print the response
            if first_chunk:
                print(f"# {response.name}: ", end="", flush=True)
                first_chunk = False
            print(f"{response}", end="", flush=True)
            thread = response.thread
        print()

    # Clean up the thread
    await thread.delete() if thread else None

await main()

# User: Plan me a day trip.

# TravelAgent: # TravelAgent: Your day trip destination isYour day trip destination is Cape Town, South Cape Town, South Africa! Here's a suggested day Africa! Here's a suggested day plan plan to explore this beautiful city to explore this beautiful city:

###:

### Morning:
- **Start your Morning:
- **Start your day with day with a visit to a visit to Table Mountain** Table Mountain**: Take the: Take the cable car or cable car or go hiking if you're go hiking if you're feeling adventurous. Enjoy feeling adventurous. Enjoy the panoramic the panoramic views of the views of the city, city, coastline coastline, and beyond, and beyond from the top of from the top of this iconic landmark.

### Midday:
- **Head this iconic landmark.

### Midday:
- **Head to the V&A Waterfront to the V&A Waterfront**: S**: Savor a delicious brunchavor a delicious brunch at one at one of the many waterfront cafes. Explore of the many waterfront cafes. Explore the shopping area, str